# Bonus — Aplicaciones de Machine Learning

Este notebook aborda tres problemas de ML con distintos enfoques:

1. **Clasificación de imágenes** — Detección de defectos en superficies metálicas industriales con una ResNet18 preentrenada.
2. **Predicción de mercado** — Predicción de retornos de la acción de Toyota con Random Forest.
3. **Riesgo crediticio** — Predicción de aprobación de préstamos con Regresión Logística.

Cada sección descarga su dataset desde Kaggle, realiza un análisis exploratorio, entrena un modelo y lo evalúa.


## Configuración

Instalación de dependencias externas e importación de todas las librerías utilizadas en el notebook.


In [ ]:
%pip install -q kagglehub

In [ ]:
import os
import glob

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    TimeSeriesSplit,
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

## 1. Clasificación de imágenes — Defectos en superficies metálicas

Se ajusta una CNN (ResNet18 preentrenada en ImageNet) para clasificar imágenes de superficies metálicas en cinco categorías de defectos. El dataset se descarga desde Kaggle y contiene splits de entrenamiento y validación con imágenes en escala de grises.

### 1.1 Descarga del dataset


In [ ]:
extract_path = kagglehub.dataset_download("tatheerabbas/synthetic-industrial-metal-surface-defects")
print("Ruta del dataset:", extract_path)

### 1.2 Carga y exploración de datos


In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_path = os.path.join(extract_path, "industrial_defect_dataset", "train")
val_path = os.path.join(extract_path, "industrial_defect_dataset", "val")

train_dataset = datasets.ImageFolder(train_path, transform=transform)
val_dataset = datasets.ImageFolder(val_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print("Clases:", train_dataset.classes)
print("Imágenes de entrenamiento:", len(train_dataset))
print("Imágenes de validación:", len(val_dataset))


### 1.3 Entrenamiento básico de ResNet18

Se ajusta una ResNet18 preentrenada durante 5 épocas. La última capa fully-connected se reemplaza para producir 5 salidas correspondientes a las categorías de defectos.


In [ ]:
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Época {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")


### 1.4 Entrenamiento con evaluación completa

El entrenamiento básico solo reporta el loss de entrenamiento. Para evaluar correctamente el rendimiento se reentrena un modelo nuevo y se registran accuracy de entrenamiento/validación, se genera un reporte de clasificación, matriz de confusión y curvas de aprendizaje.


In [ ]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
train_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    val_accuracies.append(val_acc)

    print(f"Época {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Acc:    {val_acc:.4f}")
    print("-" * 40)

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\nReporte de clasificación:\n")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=train_dataset.classes,
            yticklabels=train_dataset.classes)
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.title("Matriz de confusión")
plt.show()

plt.figure()
plt.plot(train_losses)
plt.title("Loss por época")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.show()

plt.figure()
plt.plot(train_accuracies, label="Entrenamiento")
plt.plot(val_accuracies, label="Validación")
plt.title("Accuracy por época")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


### 1.5 Optimización de hiperparámetros (ResNet18)

Se realiza una búsqueda manual sobre la tasa de aprendizaje y el optimizador. Para cada combinación se entrena la red durante 5 épocas y se registra la accuracy de validación. Finalmente se compara el rendimiento de la configuración base contra la mejor configuración encontrada.


In [ ]:
search_space = [
    {"lr": 0.001, "optimizer": "Adam"},
    {"lr": 0.0005, "optimizer": "Adam"},
    {"lr": 0.0001, "optimizer": "Adam"},
    {"lr": 0.001, "optimizer": "SGD"},
    {"lr": 0.01, "optimizer": "SGD"},
]

results = []

for config in search_space:
    model_hp = resnet18(weights=ResNet18_Weights.DEFAULT)
    model_hp.fc = nn.Linear(model_hp.fc.in_features, 5)
    model_hp = model_hp.to(device)

    if config["optimizer"] == "Adam":
        opt = optim.Adam(model_hp.parameters(), lr=config["lr"])
    else:
        opt = optim.SGD(model_hp.parameters(), lr=config["lr"], momentum=0.9)

    criterion_hp = nn.CrossEntropyLoss()

    for epoch in range(5):
        model_hp.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            loss = criterion_hp(model_hp(images), labels)
            loss.backward()
            opt.step()

    model_hp.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_hp(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    results.append({**config, "val_acc": val_acc})
    print(f"Optimizer: {config['optimizer']}, LR: {config['lr']} -> Val Acc: {val_acc:.4f}")

results_df = pd.DataFrame(results).sort_values("val_acc", ascending=False)
print("\nResultados ordenados por accuracy de validación:\n")
print(results_df.to_string(index=False))

best = results_df.iloc[0]
baseline = next(r for r in results if r["lr"] == 0.001 and r["optimizer"] == "Adam")

print(f"\nConfiguración base (Adam, lr=0.001): Val Acc = {baseline['val_acc']:.4f}")
print(f"Mejor configuración ({best['optimizer']}, lr={best['lr']}): Val Acc = {best['val_acc']:.4f}")
print(f"Diferencia: {best['val_acc'] - baseline['val_acc']:+.4f}")


## 2. Predicción de mercado — Acción de Toyota

Se utiliza un Random Forest para predecir el precio de cierre de la acción de Toyota a partir de las columnas Open, High, Low y Volume. El dataset abarca de 1980 a 2026.

### 2.1 Descarga del dataset


In [ ]:
dataset_path = kagglehub.dataset_download("omarshahrukh/toyota-stock-prices-1980-2026-historical-data")
found = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
toyota_csv = found[0]
print("Ruta del dataset:", toyota_csv)

### 2.2 Exploración de datos


In [ ]:
df_stock = pd.read_csv(toyota_csv)

print(df_stock.head())
print(df_stock.info())
print("Dimensiones:", df_stock.shape)

### 2.3 Predicción del precio de cierre (Random Forest)

Primer enfoque: predecir directamente el precio de cierre (Close) a partir de las demás columnas. La división es secuencial (sin shuffle) para respetar el orden temporal.


In [ ]:
df = pd.read_csv(toyota_csv)

print("Dimensiones:", df.shape)
print(df.head())

X = df[["Open", "High", "Low", "Volume"]]
y = df["Close"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\nMétricas de regresión")
print(f"  MAE:  {mae:.4f}")
print(f"  MSE:  {mse:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  R2:   {r2:.4f}")

plt.figure(figsize=(12, 6))
plt.plot(y_test.values, label="Real")
plt.plot(y_pred, label="Predicho")
plt.title("Precio real vs predicho")
plt.legend()
plt.show()


### 2.4 Predicción de retornos (Random Forest)

El modelo anterior se entrena con datos de los 80s y 90s pero se prueba con datos de 2010 a 2026, lo que genera que no prediga bien la escala del precio. Esto no es un problema del modelo sino del mercado: el precio absoluto cambia drásticamente en el tiempo.

La solución es predecir retornos porcentuales en lugar del precio en sí mismo, y luego extrapolar esos retornos al valor real de la acción.


In [ ]:
df = pd.read_csv(toyota_csv)

df["Date"] = pd.to_datetime(df["Date"])
df.sort_values("Date", inplace=True)

df["Return"] = df["Close"].pct_change()
df["Return_lag1"] = df["Return"].shift(1)
df["Return_lag2"] = df["Return"].shift(2)
df["Return_lag3"] = df["Return"].shift(3)
df.dropna(inplace=True)

X = df[["Return_lag1", "Return_lag2", "Return_lag3"]]
y = df["Return"]

split_index = int(len(df) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Métricas de regresión")
print(f"  MAE:  {mae:.6f}")
print(f"  RMSE: {rmse:.6f}")
print(f"  R2:   {r2:.6f}")

last_train_price = df["Close"].iloc[split_index]
predicted_prices = [last_train_price]
for r in y_pred:
    predicted_prices.append(predicted_prices[-1] * (1 + r))
predicted_prices = predicted_prices[1:]

plt.figure(figsize=(12, 6))
plt.plot(y_test.values, label="Retorno real")
plt.plot(y_pred, label="Retorno predicho")
plt.legend()
plt.title("Retorno real vs predicho")
plt.show()


### 2.5 Optimización de hiperparámetros (Random Forest)

Se aplica GridSearchCV con TimeSeriesSplit (3 folds) para buscar la mejor combinación de hiperparámetros del Random Forest. Los parámetros a optimizar son el número de árboles, la profundidad máxima y el mínimo de muestras por hoja. Se compara el rendimiento del modelo base contra el modelo optimizado usando MAE, RMSE y R2.


In [ ]:
df_opt = pd.read_csv(toyota_csv)

X_opt = df_opt[["Open", "High", "Low", "Volume"]]
y_opt = df_opt["Close"]

X_train_opt, X_test_opt, y_train_opt, y_test_opt = train_test_split(
    X_opt, y_opt, test_size=0.2, shuffle=False
)

baseline_rf = RandomForestRegressor(n_estimators=200, random_state=42)
baseline_rf.fit(X_train_opt, y_train_opt)
y_pred_base = baseline_rf.predict(X_test_opt)

mae_base = mean_absolute_error(y_test_opt, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test_opt, y_pred_base))
r2_base = r2_score(y_test_opt, y_pred_base)

param_grid = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 10, 20, 30],
    "min_samples_leaf": [1, 2, 4],
}

tscv = TimeSeriesSplit(n_splits=3)

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train_opt, y_train_opt)

print("Mejores hiperparámetros:", grid_search.best_params_)

y_pred_tuned = grid_search.best_estimator_.predict(X_test_opt)

mae_tuned = mean_absolute_error(y_test_opt, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test_opt, y_pred_tuned))
r2_tuned = r2_score(y_test_opt, y_pred_tuned)

comparison = pd.DataFrame({
    "Métrica": ["MAE", "RMSE", "R2"],
    "Baseline": [mae_base, rmse_base, r2_base],
    "Optimizado": [mae_tuned, rmse_tuned, r2_tuned],
    "Diferencia": [mae_tuned - mae_base, rmse_tuned - rmse_base, r2_tuned - r2_base],
})
print("\nComparación antes vs después del ajuste:\n")
print(comparison.to_string(index=False))

plt.figure(figsize=(12, 6))
plt.plot(y_test_opt.values, label="Real")
plt.plot(y_pred_base, label="Baseline", alpha=0.7)
plt.plot(y_pred_tuned, label="Optimizado", alpha=0.7)
plt.title("Precio real vs predicho (Baseline vs Optimizado)")
plt.legend()
plt.show()


## 3. Riesgo crediticio — Predicción de aprobación de préstamos

Se entrena un modelo de Regresión Logística para predecir si un préstamo será aprobado o rechazado. Se utiliza un pipeline de preprocesamiento que estandariza variables numéricas y codifica las categóricas con One-Hot Encoding.

### 3.1 Descarga del dataset


In [ ]:
dataset_path = kagglehub.dataset_download("sohailkhan05/loan-risk-prediction")
found = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
loan_csv = found[0]
print("Ruta del dataset:", loan_csv)

### 3.2 Exploración de datos


In [ ]:
df_loan = pd.read_csv(loan_csv)

print(df_loan.head())
print(df_loan.info())
print("Dimensiones:", df_loan.shape)

### 3.3 Entrenamiento y evaluación (Regresión Logística)

Se construye un pipeline que estandariza las variables numéricas y codifica las categóricas. Se utiliza Regresión Logística con pesos balanceados para compensar posibles desbalances de clase.


In [ ]:
df = pd.read_csv(loan_csv)

print("Dimensiones:", df.shape)
print(df.head())

df.fillna(df.mean(numeric_only=True), inplace=True)
df.fillna("Unknown", inplace=True)

X = df.drop("LoanApproved", axis=1)
y = df["LoanApproved"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))


### 3.4 Optimización de hiperparámetros (Regresión Logística)

Se aplica GridSearchCV con 5 folds para optimizar el parámetro de regularización C y el tipo de penalización de la Regresión Logística. Se compara el rendimiento del modelo base contra el optimizado usando accuracy, precisión, recall y F1-score.


In [ ]:
df_lr = pd.read_csv(loan_csv)

df_lr.fillna(df_lr.mean(numeric_only=True), inplace=True)
df_lr.fillna("Unknown", inplace=True)

X_lr = df_lr.drop("LoanApproved", axis=1)
y_lr = df_lr["LoanApproved"]

num_cols_lr = X_lr.select_dtypes(include=["int64", "float64"]).columns
cat_cols_lr = X_lr.select_dtypes(include=["object"]).columns

preprocessor_lr = ColumnTransformer([
    ("num", StandardScaler(), num_cols_lr),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_lr)
])

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.2, random_state=42
)

baseline_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=42
    ))
])
baseline_pipe.fit(X_train_lr, y_train_lr)
y_pred_base_lr = baseline_pipe.predict(X_test_lr)
acc_base = accuracy_score(y_test_lr, y_pred_base_lr)

param_grid_lr = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "classifier__penalty": ["l1", "l2"],
    "classifier__solver": ["liblinear", "saga"],
}

grid_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=42
    ))
])

grid_lr = GridSearchCV(
    grid_pipe,
    param_grid_lr,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
grid_lr.fit(X_train_lr, y_train_lr)

print("Mejores hiperparámetros:", grid_lr.best_params_)

y_pred_tuned_lr = grid_lr.best_estimator_.predict(X_test_lr)
acc_tuned = accuracy_score(y_test_lr, y_pred_tuned_lr)

print(f"\nAccuracy base (C=1, l2):     {acc_base:.4f}")
print(f"Accuracy optimizado:         {acc_tuned:.4f}")
print(f"Diferencia:                  {acc_tuned - acc_base:+.4f}")

print("\n--- Reporte base ---")
print(classification_report(y_test_lr, y_pred_base_lr))

print("--- Reporte optimizado ---")
print(classification_report(y_test_lr, y_pred_tuned_lr))
